# Giải pháp Dự đoán Sản lượng theo Tuần

## Mục tiêu

Dự báo `quantity` cho từng cặp `(location, item_id)` với các ràng buộc:

- Dữ liệu **3 tuần đầu** (`w1`, `w2`, `w3`) của tháng 12 dùng làm lịch sử.
- **Tuần 4** (`w4`) là giai đoạn cần dự đoán — tại thời điểm forecast, chưa có bất kỳ thông tin nào về `w4`.
- Metric đánh giá: **MAE** (Mean Absolute Error) — chỉ được tính **sau** khi đã có prediction.
- Bảng output cuối cùng gồm: `location`, `item_id`, `quantity_predict`, `price`.

## Nguyên tắc quan trọng

| Giai đoạn | Dữ liệu được phép dùng | Mục đích |
|-----------|------------------------|----------|
| **Forecast** | Chỉ `w1`, `w2`, `w3` | Tạo feature, xây dựng dự báo |
| **Evaluate** | `w4` (ground truth) | So sánh prediction với actual, tính MAE |

> ⚠️ `w4` **tuyệt đối không** được dùng để train, fit, hoặc xây feature dưới bất kỳ hình thức nào.

## Ý tưởng giải pháp

1. Đọc transaction gốc, tạo `week_start` bằng `dt.truncate("1w")`.
2. Ánh xạ 5 bucket lịch → 4 tuần (tuần 4 = bucket 4 + bucket 5).
3. Tạo bảng weekly demand theo `(week_no, location, item_id, quantity)`.
4. Vẽ biểu đồ tham khảo.
5. Tạo feature chỉ từ `w1`–`w3`, sinh prediction bằng baseline.
6. Dùng `w4` actual để tính MAE, đánh giá chất lượng.

### Vì sao không dùng Ridge Regression?

- Ridge là mô hình **supervised** — cần target (`y`) để `fit`.
- Trong bài toán này, target duy nhất hợp lệ là `qty_w4`, nhưng `w4` **chưa tồn tại** tại thời điểm forecast.
- Nếu dùng `qty_w4` làm `y` để train Ridge → **data leakage** (mô hình "nhìn thấy" đáp án trước khi dự đoán).
- Do repo chỉ có dữ liệu 1 tháng, không thể thiết kế temporal split sạch cho supervised model.
- **Kết luận:** Chỉ các phương pháp baseline (không cần target) mới hợp lệ trong scenario này.

## Cách thực thi

```bash
pip install polars numpy matplotlib scikit-learn notebook
jupyter notebook "Predict Solution/weekly_quantity_forecast_solution.ipynb"
```

Chọn **Run All** để chạy toàn bộ notebook.


---
## 1. Import thư viện & Cấu hình


In [9]:
from pathlib import Path
import random
import warnings

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error

# --- Cấu hình chung ---
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

SEED = 42
rng = random.Random(SEED)


# --- Tìm thư mục gốc chứa raw_data ---
def resolve_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / "raw_data").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục raw_data")


ROOT = resolve_root()
print(f"Thư mục gốc: {ROOT}")


Thư mục gốc: C:\Users\asus\Documents\HocTap\HK4\CS116 - Python Programming For ML\Pj-selling website


---
## 2. Đọc dữ liệu gốc


In [ ]:
# Đọc bảng giao dịch
transactions = pl.read_parquet(ROOT / "raw_data" / "transactions-2025-12.parquet").with_columns(
    pl.col("price").cast(pl.Float64),
    pl.col("updated_date").dt.truncate("1w").alias("week_start"),
)

# Đọc bảng sản phẩm (chỉ lấy các cột cần thiết)
items = pl.read_parquet(ROOT / "raw_data" / "items.parquet").select(
    "item_id",
    pl.col("price").cast(pl.Float64).alias("catalog_price"),
    "category_l1", "category_l2", "category_l3", "brand",
)

print(f"Số dòng giao dịch: {len(transactions):,}")
print(f"Số sản phẩm: {len(items):,}")


Số dòng giao dịch: 3,782,447
Số sản phẩm: 29,823


---
## 3. Xử lý dữ liệu – Ánh xạ tuần & Tạo bảng demand

Tháng 12/2025 có 5 bucket lịch (`2025-12-01`, `08`, `15`, `22`, `29`).
Ta ánh xạ lại thành 4 tuần:
- Tuần 1, 2, 3: giữ nguyên.
- Tuần 4 = bucket 4 + bucket 5 (gộp lại).


In [11]:
# --- Ánh xạ week_start → week_no (1–4) ---
week_map = (
    transactions.select("week_start").unique().sort("week_start")
    .with_row_count("week_idx")
    .with_columns(
        pl.when(pl.col("week_idx") >= 3).then(4)
        .otherwise(pl.col("week_idx") + 1)
        .alias("week_no")
    )
    .select("week_start", "week_no")
)

# Gắn week_no vào transactions
transactions = transactions.join(week_map, on="week_start", how="left")

# Kiểm tra phân bổ số dòng theo tuần
(
    transactions.group_by(["week_no", "week_start"])
    .agg(pl.len().alias("số_dòng"), pl.col("quantity").sum().alias("tổng_quantity"))
    .sort(["week_no", "week_start"])
)


week_no,week_start,số_dòng,tổng_quantity
u32,datetime[μs],u32,i32
1,2025-12-01 00:00:00,871693,1395555
2,2025-12-08 00:00:00,901705,1560266
3,2025-12-15 00:00:00,836120,1320356
4,2025-12-22 00:00:00,958047,1532901
4,2025-12-29 00:00:00,214882,348449


In [ ]:
# --- Tổng hợp demand theo tuần cho từng (location, item_id) ---
weekly_demand = (
    transactions
    .group_by(["week_no", "location", "item_id"])
    .agg(pl.col("quantity").sum().alias("quantity"))
    .sort(["location", "item_id", "week_no"])
)

print(f"Số dòng weekly_demand: {len(weekly_demand):,}")
weekly_demand.head()


---
## 4. Biểu đồ tham khảo

Vẽ 2 biểu đồ:
- **Trái:** 1 location, 2 sản phẩm ngẫu nhiên.
- **Phải:** 1 sản phẩm, 2 location ngẫu nhiên.

Chỉ chọn ngẫu nhiên trong các cặp có **ít nhất 2 tuần có doanh số** để biểu đồ có ý nghĩa.


In [ ]:
# --- Hàm lấy chuỗi quantity 4 tuần cho 1 cặp (location, item) ---
def lay_chuoi_tuan(location=None, item_id=None):
    """Trả về DataFrame 4 dòng (week_no 1–4) với quantity tương ứng."""
    base = pl.DataFrame({"week_no": [1, 2, 3, 4]})
    filtered = weekly_demand
    if location is not None:
        filtered = filtered.filter(pl.col("location") == location)
    if item_id is not None:
        filtered = filtered.filter(pl.col("item_id") == item_id)
    return (
        base.join(filtered.select("week_no", "quantity"), on="week_no", how="left")
        .fill_null(0).sort("week_no")
    )


# --- Tính số tuần active cho từng cặp ---
pair_activity = (
    weekly_demand.group_by(["location", "item_id"])
    .agg(pl.len().alias("active_weeks"), pl.col("quantity").sum().alias("total_qty"))
)

# Chọn 1 location có ≥ 2 sản phẩm active ≥ 2 tuần
loc_candidates = (
    pair_activity.filter(pl.col("active_weeks") >= 2)
    .group_by("location").agg(pl.len().alias("n_products"))
    .filter(pl.col("n_products") >= 2)
)
chosen_location = rng.choice(loc_candidates["location"].to_list())
chosen_products = rng.sample(
    pair_activity.filter(
        (pl.col("location") == chosen_location) & (pl.col("active_weeks") >= 2)
    )["item_id"].to_list(), 2
)

# Chọn 1 sản phẩm có ≥ 2 location active ≥ 2 tuần
prod_candidates = (
    pair_activity.filter(pl.col("active_weeks") >= 2)
    .group_by("item_id").agg(pl.len().alias("n_locations"))
    .filter(pl.col("n_locations") >= 2)
)
chosen_item = rng.choice(prod_candidates["item_id"].to_list())
chosen_locations = rng.sample(
    pair_activity.filter(
        (pl.col("item_id") == chosen_item) & (pl.col("active_weeks") >= 2)
    )["location"].to_list(), 2
)

print(f"Location: {chosen_location} | Sản phẩm: {chosen_products}")
print(f"Sản phẩm: {chosen_item} | Locations: {chosen_locations}")


# --- Vẽ biểu đồ ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Biểu đồ trái: 1 location, 2 sản phẩm
for pid in chosen_products:
    s = lay_chuoi_tuan(location=chosen_location, item_id=pid)
    axes[0].plot(s["week_no"].to_list(), s["quantity"].to_list(),
                 marker="o", linewidth=2, label=f"Sản phẩm {pid}")
axes[0].set_title(f"1 Location ({chosen_location}), 2 Sản phẩm")
axes[0].set_xlabel("Tuần"); axes[0].set_ylabel("Số lượng")
axes[0].set_xticks([1, 2, 3, 4]); axes[0].legend()

# Biểu đồ phải: 1 sản phẩm, 2 location
for lid in chosen_locations:
    s = lay_chuoi_tuan(location=lid, item_id=chosen_item)
    axes[1].plot(s["week_no"].to_list(), s["quantity"].to_list(),
                 marker="o", linewidth=2, label=f"Location {lid}")
axes[1].set_title(f"1 Sản phẩm ({chosen_item}), 2 Location")
axes[1].set_xlabel("Tuần"); axes[1].set_ylabel("Số lượng")
axes[1].set_xticks([1, 2, 3, 4]); axes[1].legend()

plt.tight_layout()
plt.show()


---
## 5. Tạo bảng Feature (chỉ từ w1–w3)

Pivot `weekly_demand` thành dạng rộng và tính thêm các đặc trưng **chỉ từ tuần 1–3**:
- `qty_mean_3w`: trung bình 3 tuần đầu.
- `qty_total_3w`: tổng 3 tuần đầu.
- `price`: ưu tiên giá trung bình từ giao dịch 3 tuần đầu, nếu không có thì dùng giá catalog.

> **Lưu ý:** Cột `qty_w4` được giữ lại trong bảng feature nhưng **chỉ phục vụ bước Evaluate** (tính MAE). Nó **không** tham gia vào bất kỳ logic forecast nào.


In [ ]:
# --- Pivot weekly_demand ra dạng rộng ---
wide = (
    weekly_demand
    .pivot(on="week_no", index=["location", "item_id"],
           values="quantity", aggregate_function="sum")
    .fill_null(0)
)
# Đảm bảo đủ 4 cột tuần
for c in ["1", "2", "3", "4"]:
    if c not in wide.columns:
        wide = wide.with_columns(pl.lit(0).alias(c))

# --- Tính giá trung bình từ 3 tuần huấn luyện ---
train_price = (
    transactions.filter(pl.col("week_no") <= 3)
    .group_by(["location", "item_id"])
    .agg(pl.col("price").mean().alias("train_price"))
)

# --- Gộp thành bảng feature ---
feature_table = (
    wide
    .rename({"1": "qty_w1", "2": "qty_w2", "3": "qty_w3", "4": "qty_w4"})
    .join(train_price, on=["location", "item_id"], how="left")
    .join(items, on="item_id", how="left")
    .with_columns(
        # Giá: ưu tiên train_price → catalog_price → 0
        pl.coalesce([pl.col("train_price"), pl.col("catalog_price"), pl.lit(0.0)]).alias("price"),
        # Trung bình 3 tuần (feature — chỉ dùng w1, w2, w3)
        ((pl.col("qty_w1") + pl.col("qty_w2") + pl.col("qty_w3")) / 3).alias("qty_mean_3w"),
        # Tổng 3 tuần (feature — chỉ dùng w1, w2, w3)
        (pl.col("qty_w1") + pl.col("qty_w2") + pl.col("qty_w3")).alias("qty_total_3w"),
    )
    .select(
        "location", "item_id", "price",
        "qty_w1", "qty_w2", "qty_w3",
        "qty_mean_3w", "qty_total_3w",
        "qty_w4",  # chỉ dùng cho evaluate, KHÔNG dùng cho forecast
    )
    .sort(["location", "item_id"])
)

print(f"Kích thước bảng feature: {feature_table.shape}")
feature_table.head(10)


---
## 6. FORECAST – Dự đoán sản lượng tuần 4 (chỉ dùng w1–w3)

Hai phương pháp baseline hợp lệ (không cần target `w4`):

| # | Phương pháp | Công thức | Dùng w4? |
|---|------------|-----------|----------|
| 1 | `baseline_last_week` | `predict = qty_w3` | ❌ Không |
| 2 | `baseline_mean_3w` | `predict = (qty_w1 + qty_w2 + qty_w3) / 3` | ❌ Không |

**Tại sao không có Ridge Regression?**
- Ridge là mô hình supervised — cần target (`y`) để `fit(X, y)`.
- Target duy nhất hợp lệ là `qty_w4`, nhưng `w4` chưa tồn tại tại thời điểm forecast.
- Dùng `qty_w4` để train → **data leakage** (mô hình nhìn thấy đáp án trước khi dự đoán).
- Với bộ dữ liệu chỉ 1 tháng, không thể thiết kế temporal split sạch cho supervised model.
- → Ridge bị **loại** vì không thể train hợp lệ trong scenario này.


In [ ]:
# ============================================================
# FORECAST PHASE — chỉ dùng w1, w2, w3. KHÔNG dùng w4.
# ============================================================

# --- Tính prediction cho 2 baseline ---
prediction_table = (
    feature_table
    .select("location", "item_id", "price")
    .with_columns(
        # Baseline 1: predict = qty_w3 (lượng tuần gần nhất)
        feature_table["qty_w3"].cast(pl.Float64).alias("pred_last_week"),
        # Baseline 2: predict = trung bình 3 tuần
        feature_table["qty_mean_3w"].cast(pl.Float64).alias("pred_mean_3w"),
    )
)

print(f"✅ Forecast hoàn tất. Số dòng: {len(prediction_table):,}")
print(f"   Cả 2 phương pháp chỉ dùng thông tin w1–w3. Không có leakage.")
prediction_table.head(10)


---
## 7. EVALUATE – Đánh giá MAE bằng actual w4

Bây giờ mới dùng `qty_w4` (ground truth) để:
- So sánh prediction với actual.
- Tính **MAE** cho từng phương pháp.
- Xếp hạng để chọn phương pháp tốt nhất.

> Đây là lần **duy nhất** `w4` xuất hiện — chỉ để evaluate, không để train.


In [ ]:
# ============================================================
# EVALUATION PHASE — dùng qty_w4 actual làm ground truth.
# ============================================================

# Lấy actual w4
actual_w4 = feature_table["qty_w4"].to_numpy().astype(float)

# Lấy predictions
pred_last_week = prediction_table["pred_last_week"].to_numpy()
pred_mean_3w = prediction_table["pred_mean_3w"].to_numpy()

# Tính MAE cho từng phương pháp
mae_scores = {
    "baseline_last_week": float(mean_absolute_error(actual_w4, pred_last_week)),
    "baseline_mean_3w":   float(mean_absolute_error(actual_w4, pred_mean_3w)),
}

# Hiển thị kết quả
print("=" * 50)
print("  KẾT QUẢ ĐÁNH GIÁ MAE (dùng actual w4)")
print("=" * 50)
mae_df = pl.DataFrame({
    "phương_pháp": list(mae_scores.keys()),
    "MAE": list(mae_scores.values()),
}).sort("MAE")
mae_df


---
## 8. Chọn phương pháp tốt nhất & Sinh bảng Output

Chọn baseline có **MAE thấp nhất** để sinh bảng dự đoán cuối cùng gồm:
- `location`
- `item_id`
- `quantity_predict`
- `price`


In [ ]:
# --- Chọn phương pháp có MAE thấp nhất ---
best = min(mae_scores, key=mae_scores.get)
print(f"Phương pháp tốt nhất: {best}  |  MAE = {mae_scores[best]:.4f}")

# --- Map tên phương pháp → tên cột prediction ---
pred_col_map = {
    "baseline_last_week": "pred_last_week",
    "baseline_mean_3w":   "pred_mean_3w",
}

# --- Sinh bảng output cuối cùng ---
output_table = (
    prediction_table
    .select(
        "location", "item_id", "price",
        pl.col(pred_col_map[best]).round(2).alias("quantity_predict"),
    )
    .sort("quantity_predict", descending=True)
)

print(f"\nSố dòng output: {len(output_table):,}")
print(f"✅ Bảng output chỉ dùng thông tin w1–w3. Không có leakage từ w4.")
output_table.head(20)


---
## Ghi chú cuối

- `price` trong output ưu tiên giá trung bình quan sát được trong 3 tuần train (`w1`–`w3`). Nếu không có thì dùng `catalog_price`.
- Nếu muốn xuất thành file riêng:

```python
output_table.write_csv(ROOT / "Predict Solution" / "week4_prediction_output.csv")
```

- Nếu sau này có dữ liệu nhiều tháng, nên:
  - Dùng backtest theo thời gian (ví dụ: train tháng 11 → predict tháng 12).
  - Thử supervised models (Ridge, Gradient Boosting, Tweedie/Poisson Regressor).
- Với bộ dữ liệu hiện tại (chỉ 1 tháng), baseline đơn giản là lựa chọn duy nhất không bị leakage.
